# Cambridge, MA — 4:1 Split-Rate LVT Model (FY2026)

**Scope:** City of Cambridge property tax levy only (Middlesex County levies no property tax).
**Reform:** 4:1 split-rate (land taxed at 4x the building rate), revenue-neutral **within each Massachusetts
property class** — the residential class and the commercial/industrial (CIP) class keep their current
levy shares; only the land/building split changes inside each class.
**Exemptions:** all preserved — fully exempt (DOR class 9xx) parcels held out; the Massachusetts
**residential exemption** ($510,208 of value for owner-occupied principal residences, FY2026) is kept as a
dollar exemption on the residential class (applied to building value first, then land, per the LVTShift
exemption hierarchy).

### Official FY2026 figures (City of Cambridge FY26 Tax Rate Letter to the City Council, Oct 6 2025; DOR-approved)
| Item | Value |
|---|---|
| Total property tax levy | $678,852,471 |
| Residential share of levy | 33.8010% |
| CIP share of levy | 66.199% |
| Residential rate | $6.67 / $1,000 |
| Commercial / industrial / personal property rate | $14.07 / $1,000 |
| Total taxable value | $73,111,556,293 |
| Residential class value | $41,167M |
| Commercial class value | $29,389M |
| Industrial class value | $538,760,573 |
| Residential exemption | $510,208 of value (30% factor) = $3,403 tax savings |

Personal property is taxed at the CIP rate but is **not** on the real-property roll, so it is outside the
model. Its value is derived as the residual: $73,111.6M − $41,167M − $29,389M − $538.8M ≈ **$2,016.8M**
(cross-check: CIP value incl. personal property × $14.07 = $449.46M vs. 66.199% × levy = $449.39M ✔).
The model's revenue gate therefore compares against the **real-property** levy: residential class levy
and CIP real-property levy (CIP levy minus personal property at $14.07).

## 1. Imports, constants, `.env`

In [1]:
import sys
import json
import os
import io
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'cambridge'
STATE_FIPS = '25'
COUNTY_FIPS = '017'           # Middlesex County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# Parcel-map identity columns (built in Section 3 at the lot level)
PARCEL_ID_COL = 'parcel_id'          # MassGIS MAP_PAR_ID of the lot
OWNER_NAME_COL = 'owner_name'        # single owner, or "Condominium (N units)" for collapsed condo lots
OWNER_ADDRESS_COL = 'site_address'   # property (site) address
PARCEL_URL_TEMPLATE = None           # Cambridge property DB deep link not verified from this environment

# Official FY2026 figures (City of Cambridge FY26 Tax Rate Letter, Oct 6 2025)
OFFICIAL_LEVY = 678_852_471
RES_SHARE = 0.338010
RES_RATE = 6.67     # $ per $1,000
CIP_RATE = 14.07    # $ per $1,000
OFFICIAL_TOTAL_VALUE = 73_111_556_293
OFFICIAL_RES_VALUE = 41_167_000_000      # reported in $M
OFFICIAL_COM_VALUE = 29_389_000_000      # reported in $M
OFFICIAL_IND_VALUE = 538_760_573
RES_EXEMPTION = 510_208

OFFICIAL_RES_LEVY = OFFICIAL_LEVY * RES_SHARE
OFFICIAL_CIP_LEVY = OFFICIAL_LEVY - OFFICIAL_RES_LEVY
PERS_PROP_VALUE = OFFICIAL_TOTAL_VALUE - OFFICIAL_RES_VALUE - OFFICIAL_COM_VALUE - OFFICIAL_IND_VALUE
OFFICIAL_CIP_REAL_LEVY = OFFICIAL_CIP_LEVY - PERS_PROP_VALUE * CIP_RATE / 1000
OFFICIAL_REAL_LEVY = OFFICIAL_RES_LEVY + OFFICIAL_CIP_REAL_LEVY
IMPLIED_EXEMPTIONS = (OFFICIAL_RES_VALUE - OFFICIAL_RES_LEVY / RES_RATE * 1000) / RES_EXEMPTION

print(f"Residential levy:         ${OFFICIAL_RES_LEVY:,.0f}")
print(f"CIP levy (incl. pers.):   ${OFFICIAL_CIP_LEVY:,.0f}")
print(f"Personal property (resid): ${PERS_PROP_VALUE:,.0f}")
print(f"CIP real-property levy:   ${OFFICIAL_CIP_REAL_LEVY:,.0f}")
print(f"Real-property levy:       ${OFFICIAL_REAL_LEVY:,.0f}")
print(f"Implied # residential exemptions: {IMPLIED_EXEMPTIONS:,.0f}")

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

Residential levy:         $229,458,924
CIP levy (incl. pers.):   $449,393,547
Personal property (resid): $2,016,795,720
CIP real-property levy:   $421,017,231
Real-property levy:       $650,476,155
Implied # residential exemptions: 13,260


## 2. Fetch / load parcel data

**Source:** MassGIS Level-3 Property Tax Parcels, Cambridge (TOWN_ID 49), **CY2025 / FY2026** roll —
the per-town download of the same data served by the `Massachusetts_Property_Tax_Parcels` FeatureServer.
`M049Assess_CY25_FY26.dbf` = one row per assessor record (condo units stacked on their lot);
`M049TaxPar_CY25_FY26.shp` = lot polygons (EPSG:26986). Joined on `LOC_ID` (100% of records match).

Massachusetts assesses at 100% of full and fair cash value (M.G.L. c.59 §38) — no assessment ratio.

| Concept | Column | Notes |
|---|---|---|
| Land value | `LAND_VAL` | $0 on condo units (MA practice) — imputed in Section 3 |
| Improvement value | `BLDG_VAL` | `OTHER_VAL` is $0 for every Cambridge record |
| Total value | `TOTAL_VAL` | = land + building |
| Use code | `USE_CODE` | DOR class, town-extended (1014/1021/0112) |
| Exemption flag | derived `full_exmp` | DOR class 9xx |
| Residential exemption | derived `res_exemption` | owner-occupancy proxy (Section 3) |
| Lot key | `LOC_ID` | condo units share their lot's LOC_ID |
| Parcel ID | `MAP_PAR_ID` / `PROP_ID` | lot / record |

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
MASSGIS_ZIP = ('https://s3.us-east-1.amazonaws.com/download.massgis.digital.mass.gov/'
               'shapefiles/l3parcels/L3_SHP_M049_CAMBRIDGE.zip')
RAW_DIR = DATA_DIR / 'raw'

if PARCEL_PATH.exists():
    gdf_raw = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf_raw):,} assessor records from cache")
else:
    shp_dir = RAW_DIR / 'L3_SHP_M049_Cambridge'
    if not shp_dir.exists():
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        r = requests.get(MASSGIS_ZIP, timeout=600)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall(RAW_DIR)
    assess = gpd.read_file(shp_dir / 'M049Assess_CY25_FY26.dbf')
    assess = pd.DataFrame(assess.drop(columns='geometry', errors='ignore'))
    polys = gpd.read_file(shp_dir / 'M049TaxPar_CY25_FY26.shp')[['LOC_ID', 'MAP_PAR_ID', 'POLY_TYPE', 'geometry']]
    assert assess['FY'].eq(2026).all()
    gdf_raw = polys.merge(assess, on='LOC_ID', how='inner')
    gdf_raw = gpd.GeoDataFrame(gdf_raw, geometry='geometry', crs=polys.crs)
    print(f"Records not matched to a lot polygon: {(~assess['LOC_ID'].isin(polys['LOC_ID'])).sum()}")
    gdf_raw.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf_raw):,} assessor records on {gdf_raw['LOC_ID'].nunique():,} lots")

for c in ['LAND_VAL', 'BLDG_VAL', 'OTHER_VAL', 'TOTAL_VAL']:
    gdf_raw[c] = pd.to_numeric(gdf_raw[c], errors='coerce').fillna(0)
print(gdf_raw[['LAND_VAL', 'BLDG_VAL', 'TOTAL_VAL']].describe().round(0))
print(f"Records with $0 land: {(gdf_raw['LAND_VAL'] == 0).sum():,}   $0 total: {(gdf_raw['TOTAL_VAL'] == 0).sum():,}")
print(f"OTHER_VAL total: ${gdf_raw['OTHER_VAL'].sum():,.0f}")

Loaded 28,812 assessor records from cache
           LAND_VAL     BLDG_VAL     TOTAL_VAL
count  2.881200e+04      28812.0  2.881200e+04
mean   9.291860e+05    2301136.0  3.230322e+06
std    9.391225e+06   17775978.0  2.487163e+07
min    0.000000e+00          0.0  0.000000e+00
25%    0.000000e+00     530100.0  6.179000e+05
50%    0.000000e+00     801100.0  9.662000e+05
75%    6.450000e+05    1207700.0  1.723450e+06
max    1.000000e+09  994917900.0  1.606143e+09
Records with $0 land: 18,554   $0 total: 2,612
OTHER_VAL total: $0


## 3. Classify and validate

### 3a. DOR class, exempt flag, property class (residential vs. CIP)
- DOR base code = first 3 digits, except a 4-char code with a leading 0 (`0112` → `112`).
- 3-char codes starting with `0` (`013`, `031`, `041`) are **mixed use**.
- **Fully exempt:** DOR class 9xx (universities, hospitals, government, Housing Authority, churches).
- **$0-value records** (the `CONDO-BLDG` master records, code 199) carry no value; they are dropped
  (their lot polygon is kept via the units stacked on it).
- **Mixed-use class split:** the roll does not carry each mixed parcel's residential %. Codes `013`
  (primarily residential) get residential share *s* and `031`/`041` (primarily commercial / industrial) get
  1 − *s*, with *s* calibrated so the roll's residential class value matches the official $41,167M.

In [3]:
code_s = gdf_raw['USE_CODE'].astype(str).str.strip().str.upper()
four0 = (code_s.str.len() >= 4) & (code_s.str[0] == '0')
gdf_raw['dor_base'] = np.where(four0, code_s.str[1:4], code_s.str[:3])
gdf_raw['use_code'] = code_s
gdf_raw['is_mixed'] = (code_s.str.len() == 3) & (code_s.str[0] == '0')
gdf_raw['full_exmp'] = (gdf_raw['dor_base'].str[0].eq('9') & ~gdf_raw['is_mixed']).astype(int)

n_zero = int((gdf_raw['TOTAL_VAL'] <= 0).sum())
recs = gdf_raw[gdf_raw['TOTAL_VAL'] > 0].copy()
print(f"Dropped {n_zero:,} $0-value records (CONDO-BLDG masters etc.)")

pure_res = recs['dor_base'].str[0].eq('1') & ~recs['is_mixed'] & (recs['full_exmp'] == 0)
mixed_res_primary = recs['use_code'].eq('013')
mixed_cip_primary = recs['is_mixed'] & ~mixed_res_primary
v_res = recs.loc[pure_res, 'TOTAL_VAL'].sum()
v_013 = recs.loc[mixed_res_primary, 'TOTAL_VAL'].sum()
v_cipmix = recs.loc[mixed_cip_primary, 'TOTAL_VAL'].sum()
MIXED_RES_SHARE = ((OFFICIAL_RES_VALUE - v_res) - v_cipmix) / (v_013 - v_cipmix)
print(f"Pure residential roll value: ${v_res/1e6:,.1f}M; mixed 013: ${v_013/1e6:,.1f}M; mixed 031/041: ${v_cipmix/1e6:,.1f}M")
print(f"Calibrated residential share: 013 -> {MIXED_RES_SHARE:.3f}, 031/041 -> {1-MIXED_RES_SHARE:.3f}")

recs['res_share'] = np.select(
    [recs['full_exmp'] == 1, pure_res, mixed_res_primary, mixed_cip_primary],
    [0.0, 1.0, MIXED_RES_SHARE, 1 - MIXED_RES_SHARE],
    default=0.0,   # DOR 3xx commercial, 4xx industrial
)
taxable = recs['full_exmp'] == 0
res_val = (recs.loc[taxable, 'TOTAL_VAL'] * recs.loc[taxable, 'res_share']).sum()
cip_val = (recs.loc[taxable, 'TOTAL_VAL'] * (1 - recs.loc[taxable, 'res_share'])).sum()
off_cip_real = OFFICIAL_COM_VALUE + OFFICIAL_IND_VALUE
print(f"\nBase reconciliation (taxable real property):")
print(f"  Residential: roll ${res_val/1e6:,.1f}M vs official ${OFFICIAL_RES_VALUE/1e6:,.1f}M (calibrated)")
print(f"  CIP real:    roll ${cip_val/1e6:,.1f}M vs official ${off_cip_real/1e6:,.1f}M  ({(cip_val/off_cip_real-1)*100:+.2f}%)")
print(f"  Total real:  roll ${(res_val+cip_val)/1e6:,.1f}M vs official ${(OFFICIAL_RES_VALUE+off_cip_real)/1e6:,.1f}M "
      f"({((res_val+cip_val)/(OFFICIAL_RES_VALUE+off_cip_real)-1)*100:+.2f}%)")
print(f"\nFully exempt records: {int(recs['full_exmp'].sum()):,}  (${recs.loc[recs['full_exmp']==1,'TOTAL_VAL'].sum()/1e9:,.1f}B)")

Dropped 2,612 $0-value records (CONDO-BLDG masters etc.)
Pure residential roll value: $39,501.8M; mixed 013: $1,714.0M; mixed 031/041: $1,123.5M
Calibrated residential share: 013 -> 0.917, 031/041 -> 0.083

Base reconciliation (taxable real property):
  Residential: roll $41,167.0M vs official $41,167.0M (calibrated)
  CIP real:    roll $29,811.6M vs official $29,927.8M  (-0.39%)
  Total real:  roll $70,978.6M vs official $71,094.8M (-0.16%)

Fully exempt records: 825  ($22.1B)


### 3b. Condominium land (the MA $0-land artifact)
Massachusetts condo units carry **$0 land** — the whole unit value is booked as building — and the lot's
`CONDO-BLDG` master record carries $0. Left alone, every condo building would read as land worth nothing
and a split-rate would cut its tax sharply (a Gate-5 artifact, cf. St. Paul's token condo land).

Units are grouped into their **building lot** (MassGIS stacks units on the lot's `LOC_ID` polygon), the lot's
land value is imputed, total value is preserved, and each unit gets land = unit value × the lot's land share,
building = remainder. Taxes are computed per unit (so the residential exemption applies per unit) and units
are summed back onto the lot for the export (Section 5).

**Imputation method — lot area × local land $/sqft** (15 nearest non-condo taxable lots ≥ 500 sqft with
assessor land), capped at 70% of the lot's total value. This is the same estimator CivicMapper uses for
these lots. The LVTShift St. Paul precedent instead imputes the neighborhood median *improvement ratio*; that
estimate is computed below as a diagnostic but **not used**: it gives every condo lot the land share of nearby
2–3-family walkups (~45%) regardless of height, and in the Gate-5 scan it ranked condos (+19% median) above
small multi-family (+4%), an economically implausible ordering for denser buildings. The $/sqft
method ties land value to lot size and location, which is what a land tax actually taxes.

Imputation runs only on lots where **no record (taxable or exempt) carries assessor land value** — a taxable
building on an exempt owner's land (university ground leases) is genuinely $0 land to the taxpayer.

In [4]:
def knn_idx(donor_xy, target_xy, k=15, chunk=500):
    # Brute-force k-nearest neighbours (numpy only; small city, no scipy dependency)
    out = []
    for i in range(0, len(target_xy), chunk):
        t = target_xy[i:i + chunk]
        d2 = ((t[:, None, :] - donor_xy[None, :, :]) ** 2).sum(axis=2)
        out.append(np.argpartition(d2, k, axis=1)[:, :k])
    return np.vstack(out)

def _xy(g):
    c = g.geometry.centroid
    return np.c_[c.x.values, c.y.values]

recs['land'] = recs['LAND_VAL'].astype(float)
recs['bldg'] = recs['BLDG_VAL'].astype(float) + recs['OTHER_VAL'].astype(float)
recs['condo_land_imputed'] = 0

lot_land_any = recs.groupby('LOC_ID')['LAND_VAL'].transform('sum') > 0
unsplit = (recs['LAND_VAL'] == 0) & (recs['bldg'] > 0)
to_impute = unsplit & ~lot_land_any
print(f"$0-land records with building value: {unsplit.sum():,}")
print(f"  on lots with no assessor land anywhere (imputed): {to_impute.sum():,}")
print(f"  on lots where another record holds land (left as-is): {(unsplit & lot_land_any).sum():,}")
print(recs.loc[unsplit & lot_land_any & (recs['full_exmp']==0), 'use_code'].value_counts().head(10))

# Donors: single-record (non-condo) taxable lots with assessor land > 0
recs_per_lot = recs.groupby('LOC_ID')['PROP_ID'].transform('size')
donor_mask = (recs_per_lot == 1) & (recs['full_exmp'] == 0) & (recs['LAND_VAL'] > 0) & (recs['bldg'] > 0)
donors = recs[donor_mask].copy()
donors['land_share'] = donors['LAND_VAL'] / donors['TOTAL_VAL']
donor_res = donors[donors['dor_base'].isin(['104', '105', '109', '111', '112', '113', '114', '125'])]
donor_com = donors[donors['dor_base'].str[0].eq('3')]

lots = recs[to_impute].copy()
lot_pts = lots.groupby('LOC_ID').agg(
    geometry=('geometry', 'first'),
    res_val=('TOTAL_VAL', lambda s: (s * lots.loc[s.index, 'res_share']).sum()),
    tot_val=('TOTAL_VAL', 'sum'),
)
lot_pts = gpd.GeoDataFrame(lot_pts, geometry='geometry', crs=recs.crs)
lot_pts['is_res_lot'] = lot_pts['res_val'] >= 0.5 * lot_pts['tot_val']

def knn_median_share(targets, donor_df, k=15):
    idx = knn_idx(_xy(donor_df), _xy(targets), k=k)
    return np.median(donor_df['land_share'].values[idx], axis=1)

lot_pts['land_share'] = np.nan
for flag, d in [(True, donor_res), (False, donor_com)]:
    m = lot_pts['is_res_lot'] == flag
    if m.any():
        lot_pts.loc[m, 'land_share'] = knn_median_share(lot_pts[m], d)
print(f"\nDonors: {len(donor_res):,} residential multi-family, {len(donor_com):,} commercial")
print(f"Imputed condo lots: {len(lot_pts):,} ({lot_pts['is_res_lot'].sum():,} residential, {(~lot_pts['is_res_lot']).sum():,} commercial)")
print(lot_pts.groupby('is_res_lot')['land_share'].describe().round(3))

# Primary: lot area x local land $/sqft (capped at 70% of the lot's value)
SQM_TO_SQFT = 10.7639
donors['psf'] = donors['LAND_VAL'] / (donors.geometry.area * SQM_TO_SQFT)
donors_psf = donors[donors.geometry.area * SQM_TO_SQFT >= 500]
idx = knn_idx(_xy(donors_psf), _xy(lot_pts), k=15)
lot_pts['psf_land'] = np.minimum(np.median(donors_psf['psf'].values[idx], axis=1) * lot_pts.geometry.area * SQM_TO_SQFT,
                                 0.7 * lot_pts['tot_val'])
lot_pts['imputed_land_share'] = lot_pts['psf_land'] / lot_pts['tot_val']
share = recs.loc[to_impute, 'LOC_ID'].map(lot_pts['imputed_land_share'])
recs.loc[to_impute, 'land'] = (recs.loc[to_impute, 'TOTAL_VAL'] * share).round(0)
recs.loc[to_impute, 'bldg'] = recs.loc[to_impute, 'TOTAL_VAL'] - recs.loc[to_impute, 'land']
recs.loc[to_impute, 'condo_land_imputed'] = 1
print(f"\nImputed condo-lot land (lot area x $/sqft, USED): ${lot_pts['psf_land'].sum()/1e9:,.2f}B "
      f"of ${lot_pts['tot_val'].sum()/1e9:,.2f}B  (median lot land share {lot_pts['imputed_land_share'].median():.3f}; "
      f"{(lot_pts['psf_land'] >= 0.7*lot_pts['tot_val']-1).sum():,} lots at the 70% cap)")
print(f"Diagnostic improvement-ratio method (NOT used): ${(lot_pts['land_share']*lot_pts['tot_val']).sum()/1e9:,.2f}B")

$0-land records with building value: 15,942
  on lots with no assessor land anywhere (imputed): 15,852
  on lots where another record holds land (left as-is): 90
use_code
102    80
325     3
013     2
140     1
Name: count, dtype: int64



Donors: 4,325 residential multi-family, 581 commercial
Imputed condo lots: 2,589 (2,541 residential, 48 commercial)
             count   mean    std    min    25%    50%    75%    max
is_res_lot                                                         
False         48.0  0.516  0.120  0.305  0.459  0.497  0.589  0.783
True        2541.0  0.450  0.064  0.250  0.412  0.450  0.494  0.659



Imputed condo-lot land (lot area x $/sqft, USED): $5.06B of $18.26B  (median lot land share 0.333; 118 lots at the 70% cap)
Diagnostic improvement-ratio method (NOT used): $8.36B


### 3c. Residential exemption (owner-occupancy proxy)
The MassGIS roll does not carry Cambridge's residential-exemption flag (Cambridge's own open-data roll does,
but it was not reachable from the modeling environment). Proxy: a residential (or mixed-use) record is treated
as owner-occupied when the owner **mails to Cambridge**, the owner is **not an entity** (LLC/INC/LP/trustee
corporations, institutions), the owner mailing **street matches the site street** with an overlapping
**house number**, and any **unit** named in the owner address matches the site unit. The proxy count is
compared to the count implied by the official levy (residential class value − residential levy / rate,
÷ $510,208). The exemption applies to the residential portion of the record only, capped at that value.

In [5]:
import re
ABBR = {'STREET': 'ST', 'AVENUE': 'AVE', 'ROAD': 'RD', 'PLACE': 'PL', 'TERRACE': 'TER', 'DRIVE': 'DR',
        'COURT': 'CT', 'LANE': 'LN', 'PARKWAY': 'PKWY', 'SQUARE': 'SQ', 'HIGHWAY': 'HWY', 'CIRCLE': 'CIR',
        'BOULEVARD': 'BLVD', 'MASS': 'MASSACHUSETTS', 'MT': 'MOUNT'}
ENTITY = re.compile(r'\b(LLC|L L C|INC|CORP|CORPORATION|LP|L P|LTD|PARTNERSHIP|COMPANY|CO|ASSOCIATES|PROPERTIES|'
                    r'REALTY|HOLDINGS|UNIVERSITY|COLLEGE|AUTHORITY|CITY OF|BANK|CHURCH|FUND|GROUP|VENTURES|'
                    r'INVESTMENTS?|MANAGEMENT|DEVELOPMENT|HOUSING)\b')

def _toks(s):
    s = re.sub(r'[^A-Z0-9#/\.\- ]', ' ', str(s or '').upper())
    return [ABBR.get(t.rstrip('.'), t.rstrip('.')) for t in s.split()]

def _nums(s):
    out = set()
    for m in re.findall(r'\d+(?:\.\d+)?(?:-\d+(?:\.\d+)?)?', str(s or '')):
        parts = m.split('-')
        try:
            lo, hi = float(parts[0]), float(parts[-1])
        except ValueError:
            continue
        out.update([parts[0], parts[-1]])
        if hi > lo and hi - lo <= 40 and lo == int(lo) and hi == int(hi):
            out.update(str(i) for i in range(int(lo), int(hi) + 1))
    return out

def _owner_unit(addr):
    m = re.search(r'(?:UNIT|APT|#|NO\.?)\s*#?\s*([A-Z0-9/\.\-]+)', str(addr or '').upper())
    return m.group(1).strip('.') if m else None

def _nu(u):
    return re.sub(r'[^A-Z0-9]', '', str(u or '').upper()).lstrip('0')

def owner_occupied_proxy(df):
    out = []
    for own_addr, own_city, owner, addr_num, street, loc in zip(
            df['OWN_ADDR'], df['OWN_CITY'], df['OWNER1'], df['ADDR_NUM'], df['FULL_STR'], df['LOCATION']):
        ok = str(own_city or '').upper().startswith('CAMBRIDGE') and not ENTITY.search(str(owner or '').upper())
        if ok:
            st = [t for t in _toks(street) if t and not t[0].isdigit()]
            ot = _toks(own_addr)
            ok = len(st) > 0 and all(t in ot for t in st[:2])
        if ok:
            ok = (len(_nums(addr_num)) == 0 or
                  len(_nums(addr_num) & _nums(' '.join(t for t in ot if t and t[0].isdigit() and '/' not in t))) > 0)
        if ok:
            ou = _owner_unit(own_addr)
            if ou is not None and loc is not None and str(loc) not in ('nan', 'None', ''):
                a, b = _nu(ou), _nu(loc)
                ok = a == b or b.endswith(a) or a.endswith(b)
        out.append(bool(ok))
    return pd.Series(out, index=df.index)

elig = (recs['full_exmp'] == 0) & (recs['res_share'] > 0)
recs['owner_occ'] = False
recs.loc[elig, 'owner_occ'] = owner_occupied_proxy(recs[elig])
recs['res_value'] = recs['TOTAL_VAL'] * recs['res_share']
recs['res_exemption'] = np.where(recs['owner_occ'], np.minimum(RES_EXEMPTION, recs['res_value']), 0.0)
print(f"Owner-occupied proxy: {recs['owner_occ'].sum():,} records   (implied by official levy: {IMPLIED_EXEMPTIONS:,.0f})")
print(f"Exempted residential value: ${recs['res_exemption'].sum()/1e9:,.2f}B   "
      f"(implied: ${(OFFICIAL_RES_VALUE - OFFICIAL_RES_LEVY/RES_RATE*1000)/1e9:,.2f}B)")
print(recs[elig].groupby('use_code')['owner_occ'].agg(['sum', 'size']).sort_values('size', ascending=False).head(12))

Owner-occupied proxy: 14,800 records   (implied by official levy: 13,260)
Exempted residential value: $7.34B   (implied: $6.77B)
           sum   size
use_code             
102       8392  13311
101       3067   3739
104       1407   2197
1021      1134   2129
105        472   1141
111         89    506
112          3    208
1014       142    180
013         15    160
031         14     85
114          1     85
0104        17     55


### 3d. Property categories
Mapped from DOR codes (town-extended Cambridge codes included). Condo lots are categorized at the lot level
after collapse (Section 5): `Condominium` when residential units dominate, else `Office / Commercial Condo`.
DOR 4xx in Cambridge is dominated by **406 HIGH-TECH** (Kendall Square R&D/lab buildings) and is labeled
`Industrial` per the DOR class.

In [6]:
CONDO_CODES = {'102', '1021', '1028', '1029', '343', '327', '339', '199'}

def categorize(code, base):
    if code in CONDO_CODES:
        return 'Condominium' if base[0] == '1' else 'Office / Commercial Condo'
    if len(code) == 3 and code[0] == '0':
        return 'Mixed Use'
    if not base.isdigit():
        return 'Other'
    b = int(base)
    if b == 101:
        return 'Single Family Residential'
    if b in (104, 105, 109):
        return 'Small Multi-Family (2-4 units)'
    if 111 <= b <= 114 or 125 <= b <= 127:
        return 'Large Multi-Family (5+ units)'
    if 130 <= b <= 132:
        return 'Vacant Land'
    if 100 <= b < 200:
        return 'Other Residential'
    if 300 <= b <= 302:
        return 'Hotel'
    if b in (336, 337, 338):
        return 'Transportation - Parking'
    if 340 <= b <= 349:
        return 'Office / Commercial Condo'
    if 320 <= b <= 329:
        return 'Retail / General Commercial'
    if 390 <= b <= 392 or 440 <= b <= 442:
        return 'Vacant Land'
    if 300 <= b < 400:
        return 'Other Commercial'
    if 400 <= b < 420:
        return 'Industrial'
    if 420 <= b < 500:
        return 'Other'          # utilities / telecom
    return 'Other'

recs['rec_category'] = [categorize(c, b) for c, b in zip(recs['use_code'], recs['dor_base'])]
recs.loc[recs['full_exmp'] == 1, 'rec_category'] = 'Exempt'
# Cambridge town-extended codes
recs.loc[recs['use_code'].isin(['1062', '1322']), 'rec_category'] = 'Vacant Land'
print(recs.loc[recs['full_exmp'] == 0, 'rec_category'].value_counts())
print("\nResidual 'Other Residential' / 'Other' codes:")
print(recs.loc[recs['rec_category'].isin(['Other Residential', 'Other']), 'use_code'].value_counts().head(15))

rec_category
Condominium                       15471
Single Family Residential          3921
Small Multi-Family (2-4 units)     3535
Large Multi-Family (5+ units)       814
Office / Commercial Condo           606
Mixed Use                           247
Vacant Land                         184
Retail / General Commercial         182
Industrial                          152
Transportation - Parking             89
Other Commercial                     76
Other Residential                    48
Other                                27
Hotel                                23
Name: count, dtype: int64

Residual 'Other Residential' / 'Other' codes:
use_code
424     21
121     15
140      9
106      7
122      5
123      4
1061     3
430      3
0102     3
450      2
422      1
1067     1
0106     1
Name: count, dtype: int64


## 4. Current tax (FY2026) and revenue validation

Each taxable record is split into a **residential piece** (value × residential share, less the residential
exemption, at $6.67) and a **CIP piece** (value × (1 − share), at $14.07).

In [7]:
tx = recs[recs['full_exmp'] == 0].copy()
pieces = []
for cls, share, rate in [('R', tx['res_share'], RES_RATE), ('C', 1 - tx['res_share'], CIP_RATE)]:
    p = pd.DataFrame({
        'rec_idx': tx.index,
        'tax_class': cls,
        'land_value': tx['land'] * share,
        'improvement_value': tx['bldg'] * share,
        'total_value': tx['TOTAL_VAL'] * share,
        'exemption': tx['res_exemption'] if cls == 'R' else 0.0,
        'millage_rate': rate,
    })
    pieces.append(p[p['total_value'] > 0])
res_pc, cip_pc = [p.reset_index(drop=True) for p in pieces]

res_rev, _, res_pc = calculate_current_tax(res_pc, tax_value_col='total_value', millage_rate_col='millage_rate',
                                           exemption_col='exemption')
cip_rev, _, cip_pc = calculate_current_tax(cip_pc, tax_value_col='total_value', millage_rate_col='millage_rate')
current_revenue = res_rev + cip_rev

for name, mod, off in [('Residential', res_rev, OFFICIAL_RES_LEVY),
                       ('CIP (real property)', cip_rev, OFFICIAL_CIP_REAL_LEVY),
                       ('Total real property', current_revenue, OFFICIAL_REAL_LEVY)]:
    print(f"{name:22s} Modeled: ${mod:,.0f}   Official: ${off:,.0f}   Gap: {(mod/off-1)*100:+.2f}%")
print(f"(Official total levy incl. personal property: ${OFFICIAL_LEVY:,.0f})")

gap_pct = (current_revenue / OFFICIAL_REAL_LEVY - 1) * 100
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"
assert abs(res_rev / OFFICIAL_RES_LEVY - 1) < 0.05 and abs(cip_rev / OFFICIAL_CIP_REAL_LEVY - 1) < 0.05

Residential            Modeled: $225,595,920   Official: $229,458,924   Gap: -1.68%
CIP (real property)    Modeled: $419,449,177   Official: $421,017,231   Gap: -0.37%
Total real property    Modeled: $645,045,097   Official: $650,476,155   Gap: -0.83%
(Official total levy incl. personal property: $678,852,471)


## 5. Split-rate model (4:1, revenue-neutral within each class)

Each class is solved separately (Rochester dual-rate precedent): the residential class keeps its current
levy, the CIP class keeps its current real-property levy. Fully-exempt records were held out above and are
not in the export. Unit-level results are then summed onto their lot (condo collapse) for export.

In [8]:
res_land_m, res_imp_m, res_new, res_out = model_split_rate_tax(
    df=res_pc, land_value_col='land_value', improvement_value_col='improvement_value',
    current_revenue=res_rev, land_improvement_ratio=LAND_IMPROVEMENT_RATIO, exemption_col='exemption')
cip_land_m, cip_imp_m, cip_new, cip_out = model_split_rate_tax(
    df=cip_pc, land_value_col='land_value', improvement_value_col='improvement_value',
    current_revenue=cip_rev, land_improvement_ratio=LAND_IMPROVEMENT_RATIO)

print(f"Residential: land {res_land_m:.4f}  building {res_imp_m:.4f} mills (current {RES_RATE})  revenue ${res_new:,.0f}")
print(f"CIP:         land {cip_land_m:.4f}  building {cip_imp_m:.4f} mills (current {CIP_RATE})  revenue ${cip_new:,.0f}")

# Revenue-weighted average millage for the single-valued export columns (Rochester precedent)
land_millage = (res_land_m * res_rev + cip_land_m * cip_rev) / current_revenue
improvement_millage = (res_imp_m * res_rev + cip_imp_m * cip_rev) / current_revenue
print(f"Revenue-weighted export millage: land {land_millage:.4f}  building {improvement_millage:.4f}")

# Pieces -> records
both = pd.concat([res_out, cip_out], ignore_index=True)
rec_tax = both.groupby('rec_idx').agg(
    current_tax=('current_tax', 'sum'), new_tax=('new_tax', 'sum'),
    taxable_land_value=('taxable_land_value', 'sum'),
    taxable_improvement_value=('taxable_improvement_value', 'sum'))
tx = tx.join(rec_tax)

# Records -> lots (condo collapse: units summed onto their LOC_ID lot polygon)
def _lot_category(g):
    if len(g) > 1 and g['use_code'].isin(CONDO_CODES).any():
        res_v = (g['TOTAL_VAL'] * g['res_share']).sum()
        return 'Condominium' if res_v >= 0.5 * g['TOTAL_VAL'].sum() else 'Office / Commercial Condo'
    return g.loc[g['TOTAL_VAL'].idxmax(), 'rec_category']

lot_cat = tx.groupby('LOC_ID').apply(_lot_category, include_groups=False).rename('PROPERTY_CATEGORY')
agg = tx.groupby('LOC_ID').agg(
    parcel_id=('MAP_PAR_ID', 'first'),
    geometry=('geometry', 'first'),
    n_records=('PROP_ID', 'size'),
    owner1=('OWNER1', 'first'),
    site_address=('SITE_ADDR', 'first'),
    land_value=('land', 'sum'), improvement_value=('bldg', 'sum'), total_value=('TOTAL_VAL', 'sum'),
    res_exemption=('res_exemption', 'sum'), n_owner_occ=('owner_occ', 'sum'),
    condo_land_imputed=('condo_land_imputed', 'max'),
    current_tax=('current_tax', 'sum'), new_tax=('new_tax', 'sum'),
    taxable_land_value=('taxable_land_value', 'sum'),
    taxable_improvement_value=('taxable_improvement_value', 'sum'),
)
gdf = gpd.GeoDataFrame(agg.join(lot_cat), geometry='geometry', crs=recs.crs).reset_index()
gdf['owner_name'] = np.where(gdf['n_records'] > 1, 'Multiple owners (' + gdf['n_records'].astype(str) + ' units)', gdf['owner1'])
gdf['full_exmp'] = 0

# Override: $0 building value -> Vacant Land
gdf.loc[gdf['improvement_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'
gdf['tax_change'] = gdf['new_tax'] - gdf['current_tax']
gdf['tax_change_pct'] = np.where(gdf['current_tax'] > 0, gdf['tax_change'] / gdf['current_tax'] * 100, np.nan)

n_exempt_lots = recs.loc[recs['full_exmp'] == 1, 'LOC_ID'].nunique() - recs.loc[recs['full_exmp'] == 1, 'LOC_ID'].isin(gdf['LOC_ID']).sum()
print(f"\nModeled lots: {len(gdf):,} (from {len(tx):,} taxable records); "
      f"{int(recs['full_exmp'].sum()):,} fully-exempt records ({n_exempt_lots:,} wholly-exempt lots) held out.")
print(f"Revenue check: current ${gdf['current_tax'].sum():,.0f}  new ${gdf['new_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY', current_tax_col='current_tax', new_tax_col='new_tax')
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact (within-class)")

Residential: land 11.6271  building 2.9068 mills (current 6.67)  revenue $225,595,920
CIP:         land 30.9080  building 7.7270 mills (current 14.07)  revenue $419,449,177
Revenue-weighted export millage: land 24.1648  building 6.0412



Modeled lots: 12,047 (from 25,375 taxable records); 825 fully-exempt records (687 wholly-exempt lots) held out.
Revenue check: current $645,045,097  new $645,045,097

cambridge — 4:1 Split-Rate Tax Impact (within-class)
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential   3914      $5,111,195        9.5%     $1,306       $1,691   24.9%      19.3%            62.2%            11.2%
Small Multi-Family (2-4 units)   3522      $3,871,971       10.0%     $1,099       $1,241   17.0%      12.9%            54.9%            12.6%
                   Condominium   2562     $-5,499,883       -7.4%    $-2,147       $1,063   15.8%      10.2%            50.4%            23.1%
 Large Multi-Family (5+ units)    808     $-4,595,461       -9.4%    $-5,687         $879    6.6%       4.9%            38.0%            15.6%
     Office / Commercial Condo    289      $5,015,696        3.4

### Gate 5 — artifact scan

In [9]:
d = gdf.copy()
d['bldg_share'] = d['improvement_value'] / d['total_value'].replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('improvement_value', lambda s: (s <= 1000).mean()),
    pct_condo_imputed=('condo_land_imputed', 'mean'),
    current_tax_M=('current_tax', lambda s: s.sum() / 1e6),
    new_tax_M=('new_tax', lambda s: s.sum() / 1e6),
).sort_values('median_pct', ascending=False)
print(scan.round(3))
print(f"\nCondo lots with imputed land: {int(d['condo_land_imputed'].sum()):,}; "
      f"residential condo-lot median change {d.loc[(d['condo_land_imputed']==1)&(d['PROPERTY_CATEGORY']=='Condominium'),'tax_change_pct'].median():+.1f}%")
print(f"Lots with $0 current tax (exemption >= value): {(d['current_tax'] <= 0).sum():,}")
d['owner_occ_lot'] = d['n_owner_occ'] > 0
print("\nSingle-record residential lots — median % change by owner-occupancy (residential exemption):")
print(d[(d['n_records'] == 1) & d['PROPERTY_CATEGORY'].isin(
    ['Single Family Residential', 'Small Multi-Family (2-4 units)', 'Large Multi-Family (5+ units)'])
      ].groupby(['PROPERTY_CATEGORY', 'owner_occ_lot'])['tax_change_pct'].median().round(1))

                                   n  median_pct  median_bldg_share  \
PROPERTY_CATEGORY                                                     
Transportation - Parking          64     118.584              0.007   
Other                             19      88.903              0.187   
Other Commercial                  74      79.266              0.245   
Vacant Land                      193      74.319              0.000   
Retail / General Commercial      175      39.734              0.485   
Office / Commercial Condo        289      27.015              0.566   
Hotel                             23      24.769              0.576   
Single Family Residential       3914      19.736              0.544   
Mixed Use                        239      14.772              0.549   
Other Residential                 37      14.747              0.456   
Small Multi-Family (2-4 units)  3522      12.885              0.557   
Condominium                     2562      10.568              0.670   
Large 

## 6. Exploration charts (optional)

In [10]:
import matplotlib
matplotlib.use('Agg')
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 Split-Rate, within class)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

### Sensitivity — how the residential exemption is allocated between land and building
The LVTShift exemption hierarchy applies dollar exemptions to **building value first**, then land (used above).
Cambridge's exemption is unusually large ($510,208 — often more than half a unit's value), so building-first
allocation leaves owner-occupants with a land-heavy base taxed at the higher land rate: they see larger
increases than otherwise-identical rental property. The alternative below allocates the exemption
**proportionally** to land and building (it keeps the same share of each unit's value exempt). This is a
policy-design choice for whoever drafts the ordinance, not a data issue; the published model uses the
repo default.

In [11]:
_alt = res_pc.copy()
_f = (1 - _alt['exemption'] / _alt['total_value'].replace(0, np.nan)).fillna(1).clip(0, 1)
_alt['land_value'] *= _f
_alt['improvement_value'] *= _f
_alt['exemption'] = 0.0
_aL, _aI, _aRev, _alt = model_split_rate_tax(df=_alt, land_value_col='land_value', improvement_value_col='improvement_value',
                                           current_revenue=res_rev, land_improvement_ratio=LAND_IMPROVEMENT_RATIO)
print(f"Proportional allocation — residential land {_aL:.4f}  building {_aI:.4f} mills")
_alt_rec = pd.concat([_alt, cip_out]).groupby('rec_idx')[['current_tax', 'new_tax']].sum()
_alt_rec = tx[['LOC_ID']].join(_alt_rec).groupby('LOC_ID').sum()
_cmp = gdf.set_index('LOC_ID')[['PROPERTY_CATEGORY', 'n_records', 'n_owner_occ']].join(_alt_rec)
_cmp['alt_pct'] = np.where(_cmp['current_tax'] > 0, (_cmp['new_tax'] / _cmp['current_tax'] - 1) * 100, np.nan)
_cmp['base_pct'] = gdf.set_index('LOC_ID')['tax_change_pct']
print(_cmp.groupby('PROPERTY_CATEGORY')[['base_pct', 'alt_pct']].median().round(1))
_cmp['oo'] = _cmp['n_owner_occ'] > 0
print(_cmp[(_cmp['n_records'] == 1) & _cmp['PROPERTY_CATEGORY'].isin(['Single Family Residential', 'Small Multi-Family (2-4 units)'])]
      .groupby(['PROPERTY_CATEGORY', 'oo'])[['base_pct', 'alt_pct']].median().round(1))

Proportional allocation — residential land 12.6860  building 3.1715 mills
                                base_pct  alt_pct
PROPERTY_CATEGORY                                
Condominium                         10.6     -5.4
Hotel                               24.8     24.8
Industrial                          -7.9     -7.9
Large Multi-Family (5+ units)        4.9     12.8
Mixed Use                           14.8     20.4
Office / Commercial Condo           27.0     27.0
Other                               88.9     88.9
Other Commercial                    79.3     79.3
Other Residential                   14.7     25.2
Retail / General Commercial         39.7     39.7
Single Family Residential           19.7     12.6
Small Multi-Family (2-4 units)      12.9     10.7
Transportation - Parking           118.6    118.6
Vacant Land                         74.3     90.2
                                      base_pct  alt_pct
PROPERTY_CATEGORY              oo                      
Single Family 

## 7. Census join, standard export, report, parcel map

`df` and `gdf` are the same lot-level frame here (the census join adds columns in place).

In [12]:
df = gdf

# Census join — OPTIONAL, must happen before export
import os
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    # No key configured — the census join is optional, so skip it cleanly.
    # The model still produces the three core tax-impact charts; only the four
    # neighborhood-equity charts are skipped. This is expected, not a failure.
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        df[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                df = match_to_census_blockgroups(df, census_gdf)
                # Merge demographic attributes onto df by std_geoid
                _demo_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct', 'total_pop']
                df = df.merge(
                    census_data[[c for c in _demo_cols if c in census_data.columns]],
                    on='std_geoid', how='left'
                )
                print(f"Census join: {df['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    df[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            df[_col] = float('nan')

# Export — df must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + self-contained interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

Census join: skipped (no CENSUS_API_KEY)
  ✓ cambridge: 12,047 rows → ../../analysis/data/cambridge.csv  [model: split_rate_4to1]


create_city_report: excluded 45 fully-exempt parcels (12,047 → 12,002 modeled).


  ✓ cambridge: 12,047 parcels → ../../analysis/maps/cambridge.parquet  [map export; 0 with record links]


  ✓ cambridge: interactive map → ../../analysis/reports/cambridge/parcel_map.html  [12,047 parcels]
Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | Real property −0.83% ($645.0M modeled vs $650.5M); residential −1.68% ($225.6M vs $229.5M); CIP real −0.37% ($419.4M vs $421.0M). Official FY26 levy $678,852,471 incl. personal property (City of Cambridge FY26 Tax Rate Letter, Oct 6 2025). |
| Base reconciliation | Taxable real property on roll $70,979M vs official $71,095M (−0.16%); CIP real −0.39%; residential calibrated via mixed-use split. |
| Parcel count | 28,812 FY2026 assessor records → 25,375 taxable records → **12,047 modeled lots** (condo units collapsed onto their lot); 825 fully-exempt records ($22.1B, 687 wholly-exempt lots) held out. |
| Census coverage (Gate 3) | N/A — no `CENSUS_API_KEY` (and Census hosts unreachable from the build environment); equity charts not produced. |
| PNGs (Gate 4) | 3 of 7 (category_impact, ten_pct_share, distribution) + parcel_map.html; map parquet parcel_id 100%, parcel_url N/A (no verified deep link). |
| Artifact scan (Gate 5) | Condo $0-land artifact fixed by lot-area × $/sqft imputation (2,574 lots, $5.06B land of $18.26B). Labs (DOR 406) building-heavy (77% bldg) → decrease, real. Parking +119%, vacant +74% (real). |
| Gate 2 note | SFR median +19.7% (avg +24.9%): explained — owner-occupied SFR +24.5% vs non-owner-occupied +8.1%, driven by building-first allocation of the $510,208 residential exemption (see sensitivity; proportional allocation gives SFR +12.6%). |
| Millage (per $1,000) | Residential land 11.6271 / bldg 2.9068 (current 6.67); CIP land 30.9080 / bldg 7.7270 (current 14.07); export revenue-weighted land 24.1648 / bldg 6.0412. |

### Limitations
- **Residential exemption** is assigned by an owner-occupancy proxy (owner mailing address = site): 14,800 records vs ~13,260 implied by the levy (exempted value $7.34B vs $6.77B implied) — residential current revenue models 1.7% low.
- **Mixed-use class split** is calibrated (013 → 91.7% residential; 031/041 → 8.3%) to the official residential class value, not taken per parcel.
- **Condo land** is imputed (lot area × local land $/sqft, capped 70%); assessor land for condo units is $0 in MA.
- **Residential exemption allocation** follows the LVTShift building-first hierarchy; the proportional alternative is in the sensitivity cell.
- **Personal property** (~$2.02B, derived as the residual of the official class values) is outside the model; CIP revenue neutrality is on real property.
- `metrics_summary.md` improvement-ratio buckets use post-exemption values, so owner-occupied units whose building value is fully exempted appear as "0% improvement" there.
